In [1]:
import torch
import numpy as np
from torch import nn, Tensor
from torch.nn import Module, Sequential, Linear, ReLU, Dropout, Flatten, CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


In [2]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size) -> None:
        super().__init__()
        self.net = Sequential(
            Flatten(),
            Linear(input_size, 128),
            ReLU(),
            Dropout(p=0.15),
            ReLU(),
            Linear(128, output_size)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [3]:
def train_model_class(train_loader, test_loader, epochs, criterion):
    model = CustomModel(784, 10)
    optmizer = Adam(model.parameters(), lr=0.001)
    for i_epochs in range(epochs):    
        model.train()
        for x, y in train_loader:
            optmizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optmizer.step()

    model.eval()
    f1_arr = []
    acc_arr = []
    with torch.no_grad():
        for x, y in test_loader:
            output = model(x)
            result = torch.argmax(output, dim=1).cpu().numpy()
            acc_s = accuracy_score(y, result)
            f1_s = f1_score(y, result, average="macro")
            cm = confusion_matrix(y, result)
            f1_arr.append(f1_s)
            acc_arr.append(acc_s)

    print(f"\nResultado modelo ----- Acc: {np.mean(acc_arr)}, F1-Score: {np.mean(f1_arr)}")


In [4]:
torch.manual_seed(42)

transform = transforms.ToTensor()

dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

batch = 32
train_size = 3_000
val_size = 1_000
unused_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, _ = random_split(
    dataset,
    [train_size, val_size, unused_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(
    train_dataset, batch_size=batch, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=batch, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=batch, shuffle=False
)

In [5]:
train_model_class(train_loader, val_loader, 200, CrossEntropyLoss())


Resultado modelo ----- Acc: 0.91796875, F1-Score: 0.8880484446109447
